# Kalman Dynamic Hedge Ratios

This notebook estimates time-varying triangular hedge ratios with a random-walk Kalman filter. The state vector is

\[
\theta_t = [\alpha_t, \beta_{1,t}, \beta_{2,t}]^\top.
\]

The observation equation is

\[
y_t = [1, x_{1,t}, x_{2,t}]\theta_t + v_t,
\]

where \(y_t\) is the target log price and \(x_{1,t}, x_{2,t}\) are hedge log prices. The residual used for diagnostics is the one-step-ahead prediction error before the filter updates on the same observation.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

from src.kalman import KalmanConfig, estimate_kalman_for_triplets, compare_kalman_residuals
from src.regression import rolling_ols
from src.database import (
    connect_database,
    initialize_database,
    store_kalman_states,
    store_kalman_residuals,
)

FIGURES = ROOT / "figures"
PROCESSED = ROOT / "data" / "processed"
DATABASE = ROOT / "data" / "triangular_stat_arb.sqlite"
SCHEMA = ROOT / "sql" / "schema.sql"

FIGURES.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)


## Data source

The notebook looks for a real `data/processed/log_prices.csv` file with date-indexed log prices. If it is not present, the notebook creates a synthetic placeholder data set only to verify that the pipeline, tables, and figures run end to end. Placeholder outputs should not be interpreted as market results.


In [ ]:
log_price_path = PROCESSED / "log_prices.csv"
PLACEHOLDER_DATA = not log_price_path.exists()

if PLACEHOLDER_DATA:
    np.random.seed(7)
    n = 220
    dates = pd.bdate_range("2023-01-02", periods=n)
    smh = 5.0 + np.cumsum(0.002 + 0.015 * np.random.normal(size=n))
    qqq = 4.2 + np.cumsum(0.001 + 0.012 * np.random.normal(size=n))
    beta_1 = 0.8 + 0.18 * np.sin(np.linspace(0, 7, n))
    beta_2 = 0.35 + 0.12 * np.cos(np.linspace(0, 5, n))
    alpha = 0.25 + 0.03 * np.sin(np.linspace(0, 4, n))
    nvda = alpha + beta_1 * smh + beta_2 * qqq + 0.015 * np.random.normal(size=n)
    log_prices = pd.DataFrame({"NVDA": nvda, "SMH": smh, "QQQ": qqq}, index=dates)
else:
    log_prices = pd.read_csv(log_price_path, parse_dates=["date"]).set_index("date").sort_index()

log_prices.head()


In [ ]:
TRIPLETS = [
    {"triplet_id": "NVDA_SMH_QQQ", "target": "NVDA", "hedge_1": "SMH", "hedge_2": "QQQ"},
]

if PLACEHOLDER_DATA:
    TRIPLETS = [{"triplet_id": "NVDA_SMH_QQQ_PLACEHOLDER", "target": "NVDA", "hedge_1": "SMH", "hedge_2": "QQQ"}]

WINDOW = 60
CONFIG = KalmanConfig(
    process_noise=2.5e-5,
    measurement_noise=1e-4,
    initial_covariance=0.25,
    initial_window=WINDOW,
)


## Kalman state estimation

The state follows a random walk. A larger process-noise value lets hedge ratios move faster. A larger measurement-noise value makes the filter trust each new price observation less. These parameters are modeling assumptions and should be tested with out-of-sample diagnostics rather than chosen to make one chart look smooth.


In [ ]:
kalman_tables = estimate_kalman_for_triplets(log_prices, TRIPLETS, config=CONFIG)
kalman_states = kalman_tables["kalman_states"]
kalman_residuals = kalman_tables["kalman_residuals"]

kalman_states.head()


In [ ]:
rolling_frames = []
for triplet in TRIPLETS:
    rolling_frames.append(
        rolling_ols(
            log_prices=log_prices,
            target_col=triplet["target"],
            hedge_cols=[triplet["hedge_1"], triplet["hedge_2"]],
            window=WINDOW,
            triplet_id=triplet["triplet_id"],
        )
    )
rolling_residuals = pd.concat(rolling_frames, ignore_index=True) if rolling_frames else pd.DataFrame()
comparison_summary = compare_kalman_residuals(kalman_residuals, rolling_residuals)
comparison_summary


## Store outputs

The SQL tables separate states from residuals. The state table keeps the filtered hedge ratios. The residual table keeps one-step prediction errors that can be passed into later diagnostics and backtests.


In [ ]:
kalman_states.to_csv(PROCESSED / "kalman_hedge_ratio_table.csv", index=False)
kalman_residuals.to_csv(PROCESSED / "kalman_residual_table.csv", index=False)
comparison_summary.to_csv(PROCESSED / "kalman_vs_rolling_residual_summary.csv", index=False)

initialize_database(DATABASE, SCHEMA)
with connect_database(DATABASE) as conn:
    store_kalman_states(conn, kalman_states, if_exists="replace")
    store_kalman_residuals(conn, kalman_residuals, if_exists="replace")

{
    "kalman_states": kalman_states.shape,
    "kalman_residuals": kalman_residuals.shape,
    "comparison_summary": comparison_summary.shape,
    "placeholder_data": PLACEHOLDER_DATA,
}


## Hedge ratio comparison

Rolling OLS fully re-estimates coefficients over a fixed window. The Kalman filter updates the previous state using the newest prediction error. The Kalman path is therefore recursive rather than a full refit at each date.


In [ ]:
plot_states = kalman_states.copy()
plot_states["date"] = pd.to_datetime(plot_states["date"])
plot_rolling = rolling_residuals.copy()
plot_rolling["date"] = pd.to_datetime(plot_rolling["date"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(plot_states["date"], plot_states["beta_1"], label="Kalman beta_1")
ax.plot(plot_states["date"], plot_states["beta_2"], label="Kalman beta_2")
ax.plot(plot_rolling["date"], plot_rolling["beta_1"], label="Rolling OLS beta_1", alpha=0.7)
ax.plot(plot_rolling["date"], plot_rolling["beta_2"], label="Rolling OLS beta_2", alpha=0.7)
ax.set_title("Hedge ratio comparison" + (" placeholder" if PLACEHOLDER_DATA else ""))
ax.set_xlabel("Date")
ax.set_ylabel("Coefficient")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIGURES / "kalman_hedge_ratio_comparison.png", dpi=160)
plt.show()


## Residual comparison

The Kalman residual is the innovation: the actual target log price minus the one-step-ahead fitted value. This makes it directly comparable to rolling OLS out-of-sample residuals.


In [ ]:
residual_plot = pd.concat(
    [
        rolling_residuals[["date", "triplet_id", "method", "residual"]],
        kalman_residuals[["date", "triplet_id", "method", "residual"]],
    ],
    ignore_index=True,
)
residual_plot["date"] = pd.to_datetime(residual_plot["date"])

fig, ax = plt.subplots(figsize=(10, 5))
for method, group in residual_plot.groupby("method"):
    ax.plot(group["date"], group["residual"], label=method)
ax.axhline(0.0, linewidth=1)
ax.set_title("Residual comparison" + (" placeholder" if PLACEHOLDER_DATA else ""))
ax.set_xlabel("Date")
ax.set_ylabel("One-step residual")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIGURES / "kalman_residual_comparison.png", dpi=160)
plt.show()


## Interpretation

A Kalman filter is not automatically better than rolling OLS. It is useful when hedge ratios are believed to evolve gradually and when a recursive update is more appropriate than repeatedly discarding and refitting a window. The comparison should focus on coefficient stability, residual behavior, sensitivity to noise assumptions, and downstream backtest robustness after costs.
